In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import holidays
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

import tensorflow as tf
import keras
from keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

import os
from tensorflow.keras.models import load_model
from keras.callbacks import EarlyStopping
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

import statsmodels.api as sm
import pickle

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


import joblib

## Import Data

In [3]:
demandNSW = pd.read_csv(r"C:\Users\user\OneDrive\Desktop\University\UNSW\Capstone Project - ZZSC9020\project\data\NSW\extracted_files\totaldemand_nsw.csv",
                 index_col=[0],
                 parse_dates=[0],
                 date_format = '%d/%m/%Y %H:%M')

print(demandNSW)

                     TOTALDEMAND REGIONID
DATETIME                                 
2010-01-01 00:00:00      8038.00     NSW1
2010-01-01 00:30:00      7809.31     NSW1
2010-01-01 01:00:00      7483.69     NSW1
2010-01-01 01:30:00      7117.23     NSW1
2010-01-01 02:00:00      6812.03     NSW1
...                          ...      ...
2021-03-17 22:00:00      7419.77     NSW1
2021-03-17 22:30:00      7417.91     NSW1
2021-03-17 23:00:00      7287.32     NSW1
2021-03-17 23:30:00      7172.39     NSW1
2021-03-18 00:00:00      7094.51     NSW1

[196513 rows x 2 columns]


In [4]:
tempNSW = pd.read_csv(r"C:\Users\user\OneDrive\Desktop\University\UNSW\Capstone Project - ZZSC9020\project\data\NSW\extracted_files\temperature_nsw.csv")
tempNSW['DATETIME'] = pd.to_datetime(tempNSW['DATETIME'], format='%d/%m/%Y %H:%M')
tempNSW.drop(['LOCATION'], axis=1, inplace=True)
print(tempNSW)

                  DATETIME  TEMPERATURE
0      2010-01-01 00:00:00         23.1
1      2010-01-01 00:01:00         23.1
2      2010-01-01 00:30:00         22.9
3      2010-01-01 00:50:00         22.7
4      2010-01-01 01:00:00         22.6
...                    ...          ...
220321 2021-03-17 23:00:00         19.1
220322 2021-03-17 23:20:00         19.0
220323 2021-03-17 23:30:00         18.8
220324 2021-03-17 23:34:00         18.8
220325 2021-03-18 00:00:00         18.6

[220326 rows x 2 columns]


In [5]:
df= pd.merge(demandNSW, tempNSW, 
                              how='left', left_on='DATETIME', right_on='DATETIME')
df.drop(['REGIONID'], axis=1, inplace=True)
# Created copy
df2 = df.copy()
print(df)

                  DATETIME  TOTALDEMAND  TEMPERATURE
0      2010-01-01 00:00:00      8038.00         23.1
1      2010-01-01 00:30:00      7809.31         22.9
2      2010-01-01 01:00:00      7483.69         22.6
3      2010-01-01 01:30:00      7117.23         22.5
4      2010-01-01 02:00:00      6812.03         22.5
...                    ...          ...          ...
196521 2021-03-17 22:00:00      7419.77         19.7
196522 2021-03-17 22:30:00      7417.91         19.5
196523 2021-03-17 23:00:00      7287.32         19.1
196524 2021-03-17 23:30:00      7172.39         18.8
196525 2021-03-18 00:00:00      7094.51         18.6

[196526 rows x 3 columns]


## Feature Engineering

In [6]:
nsw_holidays = holidays.Australia(state='NSW', years=range(2010, 2022))

# Display the NSW holidays for the specified years
for date, name in sorted(nsw_holidays.items()):
    print(f"{date}: {name}")
    
df2['Holidays'] = df2.DATETIME.apply(lambda date: nsw_holidays.get(date))
df2.holidays = df2.Holidays.fillna(0) 
# df2.Holidays.unique()
mapping = {"New Year's Day" : 1,
           'None' : 0,
           'NaN' : 0,
           'Australia Day' : 1,
           'Good Friday' : 1,
           'Easter Saturday' : 1,
           'Easter Monday' : 1,
           'ANZAC Day' : 1,
           'ANZAC Day (observed)' : 1,
           "Queen's Birthday" : 1,
           'Labour Day' : 1,
           'Christmas Day' : 1,
           'Boxing Day' : 1,
           'Boxing Day (observed)' : 1 ,
           "New Year's Day (observed)" : 1,
           'Easter Sunday' : 1, 
           'ANZAC Day; Easter Monday' : 1,
           'Christmas Day (observed)' : 1,
           'Bank Holiday' : 0
          }

df2 = df2.replace({'Holidays': mapping})

df2['Holidays'].fillna(0, inplace=True)
# df2.Holidays.unique()
df2['DATETIME'] = pd.to_datetime(df2['DATETIME'])
df2['Weekday'] = df2['DATETIME'].dt.dayofweek
df2['Days_Holidays'] = df2.apply(
    lambda row: 7 if row['Holidays'] == 1 else row['Weekday'], axis=1
)
df2.drop(['Holidays','Weekday'], axis=1, inplace=True)
print(df2)

2010-01-01: New Year's Day
2010-01-26: Australia Day
2010-04-02: Good Friday
2010-04-03: Easter Saturday
2010-04-05: Easter Monday
2010-04-25: ANZAC Day
2010-04-26: ANZAC Day (observed)
2010-06-14: Queen's Birthday
2010-08-02: Bank Holiday
2010-10-04: Labour Day
2010-12-25: Christmas Day
2010-12-26: Boxing Day
2010-12-27: Boxing Day (observed)
2011-01-01: New Year's Day
2011-01-03: New Year's Day (observed)
2011-01-26: Australia Day
2011-04-22: Good Friday
2011-04-23: Easter Saturday
2011-04-24: Easter Sunday
2011-04-25: ANZAC Day; Easter Monday
2011-06-13: Queen's Birthday
2011-10-03: Labour Day
2011-12-25: Christmas Day
2011-12-26: Boxing Day
2011-12-27: Christmas Day (observed)
2012-01-01: New Year's Day
2012-01-02: New Year's Day (observed)
2012-01-26: Australia Day
2012-04-06: Good Friday
2012-04-07: Easter Saturday
2012-04-08: Easter Sunday
2012-04-09: Easter Monday
2012-04-25: ANZAC Day
2012-06-11: Queen's Birthday
2012-10-01: Labour Day
2012-12-25: Christmas Day
2012-12-26: Box

C:\Users\user\AppData\Local\Temp\ipykernel_28176\2222107764.py:8: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df2.holidays = df2.Holidays.fillna(0)
C:\Users\user\AppData\Local\Temp\ipykernel_28176\2222107764.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df2 = df2.replace({'Holidays': mapping})
C:\Users\user\AppData\Local\Temp\ipykernel_28176\2222107764.py:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always be

                  DATETIME  TOTALDEMAND  TEMPERATURE  Days_Holidays
0      2010-01-01 00:00:00      8038.00         23.1              7
1      2010-01-01 00:30:00      7809.31         22.9              7
2      2010-01-01 01:00:00      7483.69         22.6              7
3      2010-01-01 01:30:00      7117.23         22.5              7
4      2010-01-01 02:00:00      6812.03         22.5              7
...                    ...          ...          ...            ...
196521 2021-03-17 22:00:00      7419.77         19.7              2
196522 2021-03-17 22:30:00      7417.91         19.5              2
196523 2021-03-17 23:00:00      7287.32         19.1              2
196524 2021-03-17 23:30:00      7172.39         18.8              2
196525 2021-03-18 00:00:00      7094.51         18.6              3

[196526 rows x 4 columns]


In [7]:
df2['Hour'] = df2.DATETIME.dt.hour
df2['Minute'] = df2.DATETIME.dt.minute
df2['Time'] = (df2.Hour * 60 + df2.Minute) / 60
df2['Month'] = df2.DATETIME.dt.month
df2['Year'] = df2.DATETIME.dt.year
print(df2)
print(df2.dtypes)


                  DATETIME  TOTALDEMAND  TEMPERATURE  Days_Holidays  Hour  \
0      2010-01-01 00:00:00      8038.00         23.1              7     0   
1      2010-01-01 00:30:00      7809.31         22.9              7     0   
2      2010-01-01 01:00:00      7483.69         22.6              7     1   
3      2010-01-01 01:30:00      7117.23         22.5              7     1   
4      2010-01-01 02:00:00      6812.03         22.5              7     2   
...                    ...          ...          ...            ...   ...   
196521 2021-03-17 22:00:00      7419.77         19.7              2    22   
196522 2021-03-17 22:30:00      7417.91         19.5              2    22   
196523 2021-03-17 23:00:00      7287.32         19.1              2    23   
196524 2021-03-17 23:30:00      7172.39         18.8              2    23   
196525 2021-03-18 00:00:00      7094.51         18.6              3     0   

        Minute  Time  Month  Year  
0            0   0.0      1  2010  
1  

In [8]:
# Add columns for temperature statistics over the last 3 days
df2['TEMPERATURE'] = df2['TEMPERATURE'].interpolate(method='linear', limit=4)
df2['Min_Temp_48'] = df2['TEMPERATURE'].rolling(window=48, min_periods=1).min()
df2['Max_Temp_48'] = df2['TEMPERATURE'].rolling(window=48, min_periods=1).max()
df2['Avg_Temp_48'] = df2['TEMPERATURE'].rolling(window=48, min_periods=1).mean()
df2['Avg_Temp_96'] = df2['TEMPERATURE'].rolling(window=96, min_periods=1).mean()
df2['Avg_Temp_144'] = df2['TEMPERATURE'].rolling(window=144, min_periods=1).mean()


# Add columns for demand in the last and second last period
df2['TotalDemand_1S'] = df2['TOTALDEMAND'].shift(1)
df2['TotalDemand_Second_2S'] = df2['TOTALDEMAND'].shift(2)


print(df2)
print(df2.dtypes)

                  DATETIME  TOTALDEMAND  TEMPERATURE  Days_Holidays  Hour  \
0      2010-01-01 00:00:00      8038.00         23.1              7     0   
1      2010-01-01 00:30:00      7809.31         22.9              7     0   
2      2010-01-01 01:00:00      7483.69         22.6              7     1   
3      2010-01-01 01:30:00      7117.23         22.5              7     1   
4      2010-01-01 02:00:00      6812.03         22.5              7     2   
...                    ...          ...          ...            ...   ...   
196521 2021-03-17 22:00:00      7419.77         19.7              2    22   
196522 2021-03-17 22:30:00      7417.91         19.5              2    22   
196523 2021-03-17 23:00:00      7287.32         19.1              2    23   
196524 2021-03-17 23:30:00      7172.39         18.8              2    23   
196525 2021-03-18 00:00:00      7094.51         18.6              3     0   

        Minute  Time  Month  Year  Min_Temp_48  Max_Temp_48  Avg_Temp_48  \

In [9]:
df2.isnull().sum()

DATETIME                   0
TOTALDEMAND                0
TEMPERATURE              301
Days_Holidays              0
Hour                       0
Minute                     0
Time                       0
Month                      0
Year                       0
Min_Temp_48              129
Max_Temp_48              129
Avg_Temp_48              129
Avg_Temp_96               81
Avg_Temp_144              33
TotalDemand_1S             1
TotalDemand_Second_2S      2
dtype: int64

## Additional Variables: Solar Radiation and Humidity

In [10]:
def process_datetime(df, date_col, time_col):
    # Step 1: Create a new 'DATETIME' column by combining 'Date' and 'Time'
    df['DATETIME'] = df.apply(lambda x: f"{x[date_col]} {x[time_col]}", axis=1)

    # Step 2: Drop the original 'Date' and 'Time' columns
    df.drop([date_col, time_col], axis=1, inplace=True)

    # Step 3: Handle rows with '24:00' in the 'Time' part of the 'DATETIME' column
    mask = df['DATETIME'].str.contains(' 24:00')
    df.loc[mask, 'DATETIME'] = df.loc[mask, 'DATETIME'].str.replace(' 24:00', ' 00:00')
    df.loc[mask, 'DATETIME'] = pd.to_datetime(df.loc[mask, 'DATETIME'], format='%d/%m/%Y %H:%M') + pd.Timedelta(days=1)

    # Step 4: Convert the 'DATETIME' column to datetime format
    df['DATETIME'] = pd.to_datetime(df['DATETIME'], format='%d/%m/%Y %H:%M', dayfirst=True)

    # Step 5: Drop duplicate rows based on the 'DATETIME' column
    df = df.drop_duplicates(subset=['DATETIME'])

    return df

def load_and_process_humidity_solar(file_paths, date_col='Date', time_col='Time'):
    # Load all CSV files and concatenate them into one DataFrame
    dataframes = [pd.read_csv(file_path) for file_path in file_paths]
    combined_df = pd.concat(dataframes, ignore_index=True)
    
    # Process the Date and Time columns to create a DATETIME column
    combined_df['DATETIME'] = pd.to_datetime(combined_df[date_col] + ' ' + combined_df[time_col])
    
    return combined_df

humiditysolar1 = pd.read_csv(r'C:\Users\user\OneDrive\Desktop\University\UNSW\Capstone Project - ZZSC9020\project\data\humiditysolar1.csv')
humiditysolar2 = pd.read_csv(r'C:\Users\user\OneDrive\Desktop\University\UNSW\Capstone Project - ZZSC9020\project\data\humiditysolar2.csv')
humiditysolar = pd.concat([humiditysolar1, humiditysolar2], ignore_index=True)
humiditysolar = process_datetime(humiditysolar, 'Date', 'Time')
humiditysolar = humiditysolar.rename(columns={'CHULLORA HUMID 1h average [%]': 'Humidity', 'CHULLORA SOLAR 1h average [W/m≤]': 'SolarRadiation'})
humiditysolar.dropna(inplace=True)

df3= pd.merge(df2, humiditysolar, 
                              how='left', on='DATETIME')
df3['Humidity'] = df3['Humidity'].interpolate(method='linear', limit=4)
df3['SolarRadiation'] = df3['SolarRadiation'].interpolate(method='linear', limit=4)

print(df3)
df3.isnull().sum()

                  DATETIME  TOTALDEMAND  TEMPERATURE  Days_Holidays  Hour  \
0      2010-01-01 00:00:00      8038.00         23.1              7     0   
1      2010-01-01 00:30:00      7809.31         22.9              7     0   
2      2010-01-01 01:00:00      7483.69         22.6              7     1   
3      2010-01-01 01:30:00      7117.23         22.5              7     1   
4      2010-01-01 02:00:00      6812.03         22.5              7     2   
...                    ...          ...          ...            ...   ...   
196521 2021-03-17 22:00:00      7419.77         19.7              2    22   
196522 2021-03-17 22:30:00      7417.91         19.5              2    22   
196523 2021-03-17 23:00:00      7287.32         19.1              2    23   
196524 2021-03-17 23:30:00      7172.39         18.8              2    23   
196525 2021-03-18 00:00:00      7094.51         18.6              3     0   

        Minute  Time  Month  Year  Min_Temp_48  Max_Temp_48  Avg_Temp_48  \

DATETIME                    0
TOTALDEMAND                 0
TEMPERATURE               301
Days_Holidays               0
Hour                        0
Minute                      0
Time                        0
Month                       0
Year                        0
Min_Temp_48               129
Max_Temp_48               129
Avg_Temp_48               129
Avg_Temp_96                81
Avg_Temp_144               33
TotalDemand_1S              1
TotalDemand_Second_2S       2
Humidity                 2376
SolarRadiation           2376
dtype: int64

In [11]:
df3.dropna(inplace=True)

## One Hot Encoding and Transforming Time Due to Cyclic Nature

In [12]:
# One-hot encode 'Days_Holidays' and 'Month'
df3_encoded = pd.get_dummies(df3, columns=['Days_Holidays', 'Month'])
df3_encoded['Time_sin'] = np.sin(2 * np.pi * df3_encoded['Time'] / 48)
df3_encoded['Time_cos'] = np.cos(2 * np.pi * df3_encoded['Time'] / 48)
df3_encoded.drop(['Hour','Minute','Time'], axis=1, inplace=True)
df3_encoded.set_index('DATETIME', inplace=True)
print(df3_encoded)

                     TOTALDEMAND  TEMPERATURE  Year  Min_Temp_48  Max_Temp_48  \
DATETIME                                                                        
2010-01-01 01:00:00      7483.69         22.6  2010         22.6         23.1   
2010-01-01 01:30:00      7117.23         22.5  2010         22.5         23.1   
2010-01-01 02:00:00      6812.03         22.5  2010         22.5         23.1   
2010-01-01 02:30:00      6544.33         22.4  2010         22.4         23.1   
2010-01-01 03:00:00      6377.32         22.3  2010         22.3         23.1   
...                          ...          ...   ...          ...          ...   
2021-03-17 22:00:00      7419.77         19.7  2021         16.1         22.6   
2021-03-17 22:30:00      7417.91         19.5  2021         16.1         22.6   
2021-03-17 23:00:00      7287.32         19.1  2021         16.1         22.6   
2021-03-17 23:30:00      7172.39         18.8  2021         16.1         22.6   
2021-03-18 00:00:00      709

## Split data into train/test

In [13]:
# Select features and target
X = df3_encoded[['TEMPERATURE', 'Year', 'Min_Temp_48', 'Max_Temp_48', 'Avg_Temp_48', 'Avg_Temp_96', 'Avg_Temp_144', 'TotalDemand_1S', 'TotalDemand_Second_2S', 'Humidity', 'SolarRadiation', 'Days_Holidays_0', 'Days_Holidays_1', 'Days_Holidays_2', 'Days_Holidays_3', 'Days_Holidays_4', 'Days_Holidays_5', 'Days_Holidays_6', 'Days_Holidays_7', 'Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6', 'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12', 'Time_sin', 'Time_cos']]
y = df3_encoded['TOTALDEMAND']

# Define training and testing periods
train_start_dt = '2010-01-01 00:00:00'
test_start_dt = '2019-01-01 00:00:00'

# Split data into training and testing sets
train = df3_encoded[(df3_encoded.index >= train_start_dt) & (df3_encoded.index < test_start_dt)].copy()
test = df3_encoded[df3_encoded.index >= test_start_dt].copy()

# Display the split data
print(f"Training data: {train.shape[0]} rows")
print(f"Testing data: {test.shape[0]} rows")

Training data: 155580 rows
Testing data: 38267 rows


In [19]:
%%time 
# 45 step ahead additional features with early stopping

# Features to be used for LSTM input
features = ['TOTALDEMAND', 'TEMPERATURE', 'Year', 'Time_sin', 'Time_cos', 'SolarRadiation', 'Humidity',
            'TotalDemand_1S', 'TotalDemand_Second_2S', 'Days_Holidays_0', 'Days_Holidays_1', 
            'Days_Holidays_2', 'Days_Holidays_3', 'Days_Holidays_4', 'Days_Holidays_5', 'Days_Holidays_6', 'Days_Holidays_7', 
            'Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6', 'Month_7', 'Month_8', 'Month_9', 
            'Month_10', 'Month_11','Month_12']

features_no_demand = ['TEMPERATURE', 'Year', 'Time_sin', 'Time_cos', 'SolarRadiation', 'Humidity',
            'TotalDemand_1S', 'TotalDemand_Second_2S', 'Days_Holidays_0', 'Days_Holidays_1', 
            'Days_Holidays_2', 'Days_Holidays_3', 'Days_Holidays_4', 'Days_Holidays_5', 'Days_Holidays_6', 'Days_Holidays_7', 
            'Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6', 'Month_7', 'Month_8', 'Month_9', 
            'Month_10', 'Month_11','Month_12']

# Scale TOTALDEMAND separately
scaler_demand = MinMaxScaler(feature_range=(0, 1))
train_scaled_demand = scaler_demand.fit_transform(train[['TOTALDEMAND']])
test_scaled_demand = scaler_demand.transform(test[['TOTALDEMAND']])

CPU times: total: 0 ns
Wall time: 5.57 ms


## Long Short Term Memory Model (LSTM)

In [36]:
# Normalize all features except TOTALDEMAND
scaler_lstm_feat = MinMaxScaler(feature_range=(0, 1))
train_scaled_features_lstm = scaler_lstm_feat.fit_transform(train[features].drop(columns=['TOTALDEMAND']))
test_scaled_features_lstm = scaler_lstm_feat.transform(test[features].drop(columns=['TOTALDEMAND']))

# Combine scaled TOTALDEMAND and other features
train_scaled_lstm = np.hstack((train_scaled_demand, train_scaled_features_lstm))
test_scaled_lstm = np.hstack((test_scaled_demand, test_scaled_features_lstm))

steps_ahead = 1

# Function to create sequences of data for LSTM
def create_sequences(data, sequence_length):
    X = []
    y = []
    # Adjust the loop to avoid out-of-bounds error
    for i in range(sequence_length, len(data) - steps_ahead - 1):
        X.append(data[i-sequence_length:i, :])  # Past sequence_length observations
        y.append(data[i + steps_ahead - 1 , 0])  # Predict the 45th step ahead for TOTALDEMAND
    return np.array(X), np.array(y)

# Set the sequence length
sequence_length = 144  # 3 day window 

# Create sequences for training and testing sets
X_train_lstm, y_train_lstm = create_sequences(train_scaled_lstm, sequence_length)
X_test_lstm, y_test_lstm = create_sequences(test_scaled_lstm, sequence_length)

In [37]:
# # Define the path to the pre-trained model
model_path = r'C:\Users\user\OneDrive\Desktop\University\UNSW\Capstone Project - ZZSC9020\project\lstm_1'

# Check if the file exists
if os.path.exists(model_path):
    # Load the pre-trained model
    model = load_model(model_path)
    print("Loaded pre-trained LSTM model successfully.")
else:
    pass
    # Build the LSTM model
    # model = Sequential()
    # model.add(LSTM(units=80, return_sequences=True, input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])))
    # model.add(Dropout(0.1))
    # model.add(LSTM(units=96, return_sequences=False))
    # model.add(Dropout(0.3))
    # model.add(Dense(units=10))
    # model.add(Dense(units=1)) 
    # pass

    # # Compile the model
    # model.compile(optimizer='adam', loss='mean_squared_error')

    # Define early stopping callback
    # early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    # # Train the model with early stopping
    # history = model.fit(X_train_lstm, y_train_lstm, epochs=5, batch_size=64, validation_data=(X_test_lstm, y_test_lstm), callbacks=[early_stopping])



Loaded pre-trained LSTM model successfully.


In [38]:
# Make predictions on the test set
y_pred_lstm_train = model.predict(X_train_lstm)
y_pred_lstm_test = model.predict(X_test_lstm)

In [39]:
# Rescale the predictions and actual values back to the original scale

y_train_rescaled_lstm_train = scaler_demand.inverse_transform(train_scaled_demand)
y_test_rescaled_lstm_test = scaler_demand.inverse_transform(test_scaled_demand)
y_pred_rescaled_lstm_train = scaler_demand.inverse_transform(y_pred_lstm_train )
y_pred_rescaled_lstm_test = scaler_demand.inverse_transform(y_pred_lstm_test )



In [40]:
# Evaluate the model
mae11 = mean_absolute_error(y_test_rescaled_lstm_test[146:], y_pred_rescaled_lstm_test)
mape11 = mean_absolute_percentage_error(y_test_rescaled_lstm_test[146:], y_pred_rescaled_lstm_test)
mse11 = mean_squared_error(y_test_rescaled_lstm_test[146:], y_pred_rescaled_lstm_test)
print(f'LSTM Model \n 9 features, 45 step ahead \n 50 epochs with early stopping \n Random Search result optimised')
print(f'MAE: {mae11}')
print(f'MAPE: {mape11}')
print(f'MSE: {mse11}')

LSTM Model 
 9 features, 45 step ahead 
 50 epochs with early stopping 
 Random Search result optimised
MAE: 342.2507351672962
MAPE: 0.043644799595705844
MSE: 181191.02733306412


## XGBOOST

In [15]:
# Function to create lagged features
def create_lagged_features(data, target_col, lags):
    df = data.copy()
    for lag in range(1, lags + 1):
        df[f'{target_col}_lag_{lag}'] = df[target_col].shift(lag)
    return df

# Create lagged features for TOTALDEMAND (45 lagged steps)
lags = 1
train_lagged_xgb = create_lagged_features(train, 'TOTALDEMAND', lags=lags)
test_lagged_xgb = create_lagged_features(test, 'TOTALDEMAND', lags=lags)

# # Define the list of additional features
# additional_features = ['TEMPERATURE', 'Year', 'Time_sin', 'Time_cos', 'SolarRadiation', 'Humidity',
#                        'TotalDemand_1S', 'TotalDemand_Second_2S', 'Days_Holidays_0', 'Days_Holidays_1', 
#                        'Days_Holidays_2', 'Days_Holidays_3', 'Days_Holidays_4', 'Days_Holidays_5', 
#                        'Days_Holidays_6', 'Days_Holidays_7', 'Month_1', 'Month_2', 'Month_3', 'Month_4', 
#                        'Month_5', 'Month_6', 'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 
#                        'Month_12']

# Combine lagged features and additional features for training
features_xgb = [f'TOTALDEMAND_lag_{i}' for i in range(1, lags+1)] + features_no_demand
X_train_xgb = train_lagged_xgb[features_xgb].values
X_test_xgb = test_lagged_xgb[features_xgb].values

# Scale the features separately
scaler_xgb = MinMaxScaler(feature_range=(0, 1))
X_train_scaled_xgb = scaler_xgb.fit_transform(X_train_xgb)
X_test_scaled_xgb = scaler_xgb.transform(X_test_xgb)

In [16]:
# Train XGBoost Model
xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='hist',
    device ='cuda'  # Enables GPU training
)

# Train the model with early stopping
xgb_model.fit(X_train_scaled_xgb, train_scaled_demand.ravel(), verbose=False)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1000, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [17]:
# Make Predictions
y_pred_scaled_xgb_test = xgb_model.predict(X_test_scaled_xgb)
y_pred_scaled_xgb_train = xgb_model.predict(X_train_scaled_xgb)

# Rescale the predictions back to the original scale
y_pred_rescaled_boost_train = scaler_demand.inverse_transform(y_pred_scaled_xgb_train.reshape(-1, 1))
y_pred_rescaled_boost_test = scaler_demand.inverse_transform(y_pred_scaled_xgb_test.reshape(-1, 1))
y_test_rescaled_boost = scaler_demand.inverse_transform(test_scaled_demand)

# Evaluate the model on original scale
mae_rescaled = mean_absolute_error(y_test_rescaled_boost, y_pred_rescaled_boost_test)
mse_rescaled = mean_squared_error(y_test_rescaled_boost, y_pred_rescaled_boost_test)
mape_rescaled = mean_absolute_percentage_error(y_test_rescaled_boost, y_pred_rescaled_boost_test)

# Output the results
print(f'XGBoost Model with Additional Features Evaluation on Rescaled Data:')
print(f'Rescaled MAE: {mae_rescaled}')
print(f'Rescaled MSE: {mse_rescaled}')
print(f'Rescaled MAPE: {mape_rescaled}')

c:\Users\user\OneDrive\Desktop\University\UNSW\Capstone Project - ZZSC9020\project\.venv\lib\site-packages\xgboost\core.py:158: UserWarning: [00:56:57] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


XGBoost Model with Additional Features Evaluation on Rescaled Data:
Rescaled MAE: 54.7145866823691
Rescaled MSE: 6245.136176861396
Rescaled MAPE: 0.007021567349314363


## Sarimax

In [15]:
# Define exogenous features and target variables
features_sar = ['TEMPERATURE', 'SolarRadiation', 'Humidity']
y_train_sarimax = train['TOTALDEMAND'].values
y_test_sarimax = test['TOTALDEMAND'].values

# Exogenous variables for SARIMAX
X_train_sarimax = train[features_sar]
X_test_sarimax = test[features_sar]

# Scale the features (exogenous variables)
scaler_sarimax = MinMaxScaler(feature_range=(0, 1))
X_train_scaled_sarimax = scaler_sarimax.fit_transform(X_train_sarimax)
X_test_scaled_sarimax = scaler_sarimax.transform(X_test_sarimax)

# Define and fit the SARIMAX model with the training data
sarimax_model = sm.tsa.SARIMAX(
    y_train_sarimax,                       # Dependent variable (e.g., electricity demand)
    exog=X_train_scaled_sarimax,           # Exogenous variables (e.g., temperature)
    order=(1, 1, 1),                       # ARIMA (p, d, q) parameters
    seasonal_order=(1, 1, 1, 48),          # Seasonal (P, D, Q, m) with m=48 (daily seasonality)
    enforce_stationarity=False,            # Allow non-stationarity
    enforce_invertibility=False            # Allow invertibility issues
)

# Fit the model on the training data
sarimax_result = sarimax_model.fit()

# # # # Save the SARIMAX model using joblib with compression
# sarimax_results = sarimax_model.fit(disp=False)


# Load the SARIMAX model
# sarimax_results = joblib.load('sarimax_model_compressed.pkl')

In [23]:
joblib.dump(sarimax_results, 'sarimax_model_compressed.pkl', compress=3)

In [ ]:
# Forecasting on the future data (next 15,000 steps)
# Ensure that X_test_scaled_sarimax contains the future exogenous data (temperature) for the prediction period

forecast_steps = 15000  # Predict for 15,000 steps into the future
y_forecast = sarimax_result.predict(
    start=len(y_train_sarimax),            # Start predicting where the training data ends
    end=len(y_train_sarimax) + forecast_steps - 1,  # End prediction at 15,000 steps ahead
    exog=X_test_scaled_sarimax             # Provide future exogenous data for the test period
)

# y_forecast will contain the predicted values for the future 15,000 steps
print(y_forecast)

In [25]:
# Use the fitted model to forecast for the entire test set
n_forecast_steps = len(y_test_sarimax)  # Number of steps to forecast (size of test set)

# Forecast the entire test set
y_pred_sarimax_test = sarimax_results.get_forecast(steps=n_forecast_steps, exog=X_test_scaled_sarimax)

# Get the predicted mean values
y_pred_sarimax_test_mean = y_pred_sarimax_test.predicted_mean

# Convert to a NumPy array if necessary
y_pred_sarimax_test_mean = np.array(y_pred_sarimax_test_mean)

In [ ]:
# Predict for the entire test set
y_pred_sarimax_test = sarimax_results.forecast(steps=len(X_test_scaled_sarimax), exog=X_test_scaled_sarimax)
y_pred_sarimax_train = sarimax_results.forecast(steps=len(X_train_scaled_sarimax),exog=X_train_scaled_sarimax)

# Calculate evaluation metrics
mae_sarimax = mean_absolute_error(y_test_sarimax, y_pred_sarimax_test_mean)
mse_sarimax = mean_squared_error(y_test_sarimax, y_pred_sarimax_test_mean)
mape_sarimax = mean_absolute_percentage_error(y_test_sarimax, y_pred_sarimax_test_mean)

# Output the results
print(f"SARIMAX 1-Step Ahead Forecast Evaluation:")
print(f"MAE: {mae_sarimax}")
print(f"MSE: {mse_sarimax}")
print(f"MAPE: {mape_sarimax}")

In [ ]:
y_pred_sarimax_test

In [ ]:
df_trial = pd.DataFrame({
    'preds':y_pred_sarimax_test,
    'true':y_test_sarimax
})
df_trial

In [28]:
from pmdarima import auto_arima

# Convert your arrays to float32
y_train_sarimax_32 = np.array(y_train_sarimax, dtype=np.float32)
X_train_scaled_sarimax_32 = np.array(X_train_scaled_sarimax, dtype=np.float32)


# Run auto_arima to find the best SARIMAX model
stepwise_model = auto_arima(y_train_sarimax_32, exogenous=X_train_scaled_sarimax_32, seasonal=True, m=12,
                            start_p=0, start_q=0, max_p=3, max_q=3, start_P=0, start_Q=0, 
                            max_P=3, max_Q=3, d=1, D=1, trace=True, error_action='ignore', 
                            suppress_warnings=True, stepwise=True)

# Print best model
print(stepwise_model.summary())

Performing stepwise search to minimize aic
 ARIMA(0,1,0)(0,1,0)[12]             : AIC=2265694.729, Time=2.83 sec
 ARIMA(1,1,0)(1,1,0)[12]             : AIC=2019755.362, Time=88.99 sec
 ARIMA(0,1,1)(0,1,1)[12]             : AIC=inf, Time=97.97 sec


MemoryError: Unable to allocate 233. MiB for an array with shape (14, 14, 155580) and data type float64

In [21]:


# sarimax_results = sarimax_model.fit(disp=False)

# # Predict 45 steps ahead (multi-step forecast)
# # We provide the exogenous variables for 45 steps ahead from the test data
# forecast_steps = 45
# y_pred_sarimax = sarimax_results.forecast(steps=forecast_steps, exog=X_test_sarimax[:forecast_steps])

# # Evaluate the predictions
# y_true_sarimax = y_test_sarimax[:forecast_steps]  # True values for the first 45 steps

# # Calculate evaluation metrics
# mae_sarimax = mean_absolute_error(y_true_sarimax, y_pred_sarimax)
# mse_sarimax = mean_squared_error(y_true_sarimax, y_pred_sarimax)
# mape_sarimax = mean_absolute_percentage_error(y_true_sarimax, y_pred_sarimax)

# # Output the results
# print(f"SARIMAX 45-Step Ahead Forecast Evaluation:")
# print(f"MAE: {mae_sarimax}")
# print(f"MSE: {mse_sarimax}")
# print(f"MAPE: {mape_sarimax}")


## cnn

In [27]:
# # Define the list of additional features
# additional_features = ['TEMPERATURE', 'Year', 'Time_sin', 'Time_cos', 'SolarRadiation', 'Humidity',
#                        'TotalDemand_1S', 'TotalDemand_Second_2S', 'Days_Holidays_0', 'Days_Holidays_1', 
#                        'Days_Holidays_2', 'Days_Holidays_3', 'Days_Holidays_4', 'Days_Holidays_5', 
#                        'Days_Holidays_6', 'Days_Holidays_7', 'Month_1', 'Month_2', 'Month_3', 'Month_4', 
#                        'Month_5', 'Month_6', 'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 
#                        'Month_12']

# Combine lagged features and additional features for training

X_train_cnn = train[features_no_demand].values
X_test_cnn = test[features_no_demand].values

In [28]:
# X scaler (for independent variables)
scaler_X = MinMaxScaler()
X_train_cnn = scaler_X.fit_transform(X_train_cnn)
X_test_cnn = scaler_X.transform(X_test_cnn)

# Y scaler (for dependent variable)
scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(train_scaled_demand)
y_test_scaled = scaler_y.transform(test_scaled_demand)

# Reshape X for 1D-CNN (samples, timesteps, features)
X_train_cnn = X_train_cnn.reshape((X_train_cnn.shape[0], X_train_cnn.shape[1], 1))
X_test_cnn = X_test_cnn.reshape((X_test_cnn.shape[0], X_test_cnn.shape[1], 1))

In [31]:
# Build 1D CNN

# Step 1: Define the 1D CNN model with the specified architecture
def create_cnn_model():
    model = Sequential()
    
    # Layer 1: Conv1D with 32 filters, kernel size 2, ReLU activation
    model.add(Conv1D(filters=32, kernel_size=2, activation='relu', input_shape=(X_train_cnn.shape[1], 1)))    
    # Layer 2: Conv1D with 64 filters, kernel size 2, ReLU activation
    model.add(Conv1D(filters=64, kernel_size=2, activation='relu'))    
    # Layer 3: Conv1D with 128 filters, kernel size 2, ReLU activation
    model.add(Conv1D(filters=128, kernel_size=2, activation='relu'))    
    # Layer 4: Conv1D with 256 filters, kernel size 2, ReLU activation
    model.add(Conv1D(filters=256, kernel_size=2, activation='relu'))    
    # Max Pooling layer with pool size 3
    model.add(MaxPooling1D(pool_size=3))    
    # Flatten the output to feed it into Dense layers
    model.add(Flatten())    
    # Dense layer 1 with 400 nodes
    model.add(Dense(400, activation='relu'))
    # model.add(Dropout(0.5))    
    # Dense layer 2 with 400 nodes
    model.add(Dense(400, activation='relu'))
    # model.add(Dropout(0.2))    
    # Dense layer 3 with 200 nodes
    model.add(Dense(200, activation='relu'))
    # Dense layer 3 with 100 nodes
    model.add(Dense(100, activation='relu'))
    # Dense layer 4 with 50 nodes
    # Output layer with 1 node (for regression)
    model.add(Dense(1))
    
    # Compile the model with Adam optimizer and a learning rate of 0.0001
    optimizer = Adam(learning_rate=0.0001)
    model.compile(optimizer=optimizer, loss='mean_squared_error', metrics=['mae'])
    
    return model

# Step 2: Create the model
model_test = create_cnn_model()

# Step 3: Define early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Step 4: Train the model with early stopping
Test_model = model_test.fit(X_train_cnn, y_train_scaled, 
                    epochs=5, 
                    batch_size=128, 
                    validation_split=0.2, 
                    callbacks=[early_stopping], 
                    verbose=1)

Epoch 1/5
973/973 [==============================] - 7s 7ms/step - loss: 0.0024 - mae: 0.0230 - val_loss: 3.0586e-04 - val_mae: 0.0136
Epoch 2/5
973/973 [==============================] - 6s 6ms/step - loss: 1.4526e-04 - mae: 0.0090 - val_loss: 3.5057e-04 - val_mae: 0.0160
Epoch 3/5
973/973 [==============================] - 6s 6ms/step - loss: 1.1127e-04 - mae: 0.0080 - val_loss: 2.2346e-04 - val_mae: 0.0122
Epoch 4/5
973/973 [==============================] - 6s 6ms/step - loss: 9.6135e-05 - mae: 0.0075 - val_loss: 1.1928e-04 - val_mae: 0.0081
Epoch 5/5
973/973 [==============================] - 6s 6ms/step - loss: 8.4958e-05 - mae: 0.0071 - val_loss: 1.0640e-04 - val_mae: 0.0078


In [33]:
# Step 5: Evaluate the model on the test set
test_loss, test_mae = model_test.evaluate(X_test_cnn, y_test_scaled)
print(f"Test Loss: {test_loss}, Test MAE: {test_mae}")

from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_squared_log_error, mean_absolute_percentage_error
y_pred_scaled_test_cnn = model_test.predict(X_test_cnn)
y_pred_scaled_train_cnn = model_test.predict(X_train_cnn)
y_pred_original_scale = scaler_y.inverse_transform(y_pred_scaled_test_cnn )

# Inverse transform y_test for comparison
y_test_original_scale = scaler_y.inverse_transform(y_test_scaled)
y_train_original_scale = scaler_y.inverse_transform(y_train_scaled)

# Evaluate the model using MAE or other metrics
mae = mean_absolute_error(y_test_original_scale, y_pred_original_scale)
mse = mean_squared_error(y_test_original_scale, y_pred_original_scale)
rmse = np.sqrt(mse)
mape = mean_absolute_percentage_error(y_test_original_scale, y_pred_original_scale)
msle = mean_squared_log_error(y_test_original_scale, y_pred_original_scale)
# Print all the metrics
print(f'MSE: {mse}')
print(f'RMSE: {rmse}')
print(f'MAE: {mae}')
print(f'MAPE: {mape}')
print(f'MSLE: {msle}')

# model.save('Test_model/')

1196/1196 [==============================] - 4s 3ms/step - loss: 2.6777e-04 - mae: 0.0122
Test Loss: 0.00026776877348311245, Test MAE: 0.012183692306280136
MSE: 0.0002677689407742081
RMSE: 0.016363646927693352
MAE: 0.012183685841955439
MAPE: 0.054522175738218215
MSLE: 0.0001620415700626815


In [35]:
y_pred_scaled_train_cnn

array([[0.25367254],
       [0.21868335],
       [0.1813045 ],
       ...,
       [0.29153   ],
       [0.2852203 ],
       [0.27006403]], dtype=float32)

## Ensemble

### Origional Attemp

In [ ]:
import numpy as np

# Ensure both predictions are NumPy arrays
y_pred_rescaled_boost = np.array(y_pred_rescaled_boost)
y_pred_rescaled_lstm = np.array(y_pred_rescaled_lstm)


y_pred_combined = y_pred_rescaled_boost[190:] * 0.9 + y_pred_rescaled_lstm * 0.1

mae_rescaled = mean_absolute_error(y_test_rescaled_boost[190:], y_pred_combined)
mse_rescaled = mean_squared_error(y_test_rescaled_boost[190:], y_pred_combined )
mape_rescaled = mean_absolute_percentage_error(y_test_rescaled_boost[190:], y_pred_combined )

# Output the results
print(f'XGBoost Model with Additional Features Evaluation on Rescaled Data:')
print(f'Rescaled MAE: {mae_rescaled}')
print(f'Rescaled MSE: {mse_rescaled}')
print(f'Rescaled MAPE: {mape_rescaled}')


In [ ]:
y_pred_rescaled_lstm.shape, y_pred_rescaled_boost[190:].shape, y_test_rescaled_boost[190:].shape

### XGBOOST MODEL

In [55]:
y_pred_rescaled_boost_train[146:].shape, y_pred_rescaled_lstm_train.shape, y_pred_scaled_train_cnn[146:].shape, y_train_rescaled_lstm_train[146:].shape

((155434, 1), (155434, 1), (155434, 1), (155434, 1))

In [58]:
df_ensemble_train = pd.DataFrame({
    'boost_preds': y_pred_rescaled_boost_train[146:].ravel(),  # Ensure 1D array
    'lstm_preds': y_pred_rescaled_lstm_train.ravel(),
    'cnn_preds': y_pred_scaled_train_cnn[146:].ravel(),                    # Ensure 1D array
    'true_vals': y_train_rescaled_lstm_train[146:].ravel()     # Ensure 1D array
})

df_ensemble_test = pd.DataFrame({
    'boost_preds': y_pred_rescaled_boost_test[146:].ravel(),  # Ensure 1D array
    'lstm_preds': y_pred_rescaled_lstm_test.ravel(), 
    'cnn_preds': y_pred_scaled_test_cnn[146:].ravel(),         # Ensure 1D array
    'true_vals': y_test_rescaled_lstm_test[146:].ravel()     # Ensure 1D array
})

# y_train_rescaled_lstm_train = scaler_demand.inverse_transform(train_scaled_demand)
#  = scaler_demand.inverse_transform(test_scaled_demand)
# y_pred_rescaled_lstm_train = scaler_demand.inverse_transform(y_pred_lstm_train )
# y_pred_rescaled_lstm_test = scaler_demand.inverse_transform(y_pred_lstm_test )




In [59]:
X_train_ens = df_ensemble_train[['boost_preds','lstm_preds','cnn_preds']]
Y_train_ens = df_ensemble_train['true_vals']

In [60]:
# Step 3: Initialize and train the XGBoost model
xgb_model_ens = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, random_state=42)

# Fit the model
xgb_model_ens.fit(X_train_ens, Y_train_ens)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [62]:
ens_preds = xgb_model_ens.predict(df_ensemble_test[['boost_preds','lstm_preds','cnn_preds']])

In [63]:
mae_rescaled = mean_absolute_error(df_ensemble_test['true_vals'], ens_preds)
mse_rescaled = mean_squared_error(df_ensemble_test['true_vals'], ens_preds )
mape_rescaled = mean_absolute_percentage_error(df_ensemble_test['true_vals'], ens_preds )

# Output the results
print(f'Ensemble Model with Additional Features Evaluation on Rescaled Data:')
print(f'Rescaled MAE: {mae_rescaled}')
print(f'Rescaled MSE: {mse_rescaled}')
print(f'Rescaled MAPE: {mape_rescaled}')

Ensemble Model with Additional Features Evaluation on Rescaled Data:
Rescaled MAE: 55.20521062353263
Rescaled MSE: 6032.453957523826
Rescaled MAPE: 0.007127345238942941
